# 035 — Programación probabilística y causalidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** Tratados: `(60+18)/120 = 0.650`. No tratados: `(10+85)/120 ≈ 0.792`. El agregado sugiere que el tratamiento *perjudica* (65 % < 79 %).

**E2.** Graves: tratados `0.60` vs no tratados `0.50` → el tratamiento ayuda (+10 pts). Leves: `0.90` vs `0.85` → también ayuda (+5 pts). La inversión aparece porque los graves (que se curan menos de base) reciben el tratamiento con mucha más frecuencia: `Gravedad` causa tanto el tratamiento como el resultado — confusor de libro.

**E3.** `P(cura|do(tratar)) = 0.60·0.5 + 0.90·0.5 = 0.750`. `P(cura|do(no)) = 0.50·0.5 + 0.85·0.5 = 0.675`. Efecto causal: **+7.5 puntos a favor de tratar** — lo contrario del agregado de E1. El ajuste pondera cada estrato por su peso poblacional (0.5/0.5), no por quién eligió tratarse.

**E4.**
```python
def modelo(do_tratar=None):
    grave = bernoulli(0.5)
    t = bernoulli(0.83 if grave else 0.17) if do_tratar is None else do_tratar
    p_cura = {(1,1):0.60, (1,0):0.50, (0,1):0.90, (0,0):0.85}[(grave, t)]
    return bernoulli(p_cura)
```
La línea a cortar es la asignación de `t`: `do(tratar)` reemplaza el mecanismo `t ~ f(grave)` por la constante `t = 1`. Eso es exactamente lo que distingue intervenir de condicionar.


In [ ]:
result = run_lab("probability", seed=35)
assert result["kind"] == "probability"
assert result["evidence"]
show(result)


In [ ]:
p = {("g",1): 0.60, ("g",0): 0.50, ("l",1): 0.90, ("l",0): 0.85}
pg = 0.5
asoc_t = (60+18)/120
asoc_n = (10+85)/120
do_t = p[("g",1)]*pg + p[("l",1)]*(1-pg)
do_n = p[("g",0)]*pg + p[("l",0)]*(1-pg)
print(f"E1 asociacion: tratado={asoc_t:.3f} no={asoc_n:.3f}")
print(f"E3 causal:     do(t)={do_t:.3f} do(no)={do_n:.3f} efecto={do_t-do_n:+.3f}")


## Reflexión

1. En el modelo del laboratorio, ¿qué consulta es asociacional y cuál requeriría una intervención real o un supuesto causal explícito para responderse?
2. ¿Por qué ningún volumen de datos observacionales, por sí solo, permite subir del peldaño 1 al 2 de la escalera? ¿Qué información extra hace falta?
3. Da un ejemplo donde condicionar por una variable (un colisionador, clase 027) *cree* una correlación espuria en lugar de eliminarla. ¿Qué implica para "controlar por todo"?
